# **Capstone Project: A Bi-Objective Evolutionary Approach to Feature Selection for Customer Value Prediction in Fintech**

# *Data Cleaning*

## MBAI 5600G: Applied Integrative Analytics Capstone Project

### Group 7: Brennan Mason & Mohammad Shah
---

## Environment Setup

In [ ]:
# Specify base path to local directory
BASE_PATH = "/content/drive/Shareddrives/MBAI Capstone S S26 Group 7/"

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

Mounted at /content/drive


## Data Ingestion & Inspection

In [ ]:
import numpy as np
import pandas as pd

# Define path
path = BASE_PATH + "p2p-customer-value-prediction/data/interim/sampled_loans.csv"

# Load data
loans = pd.read_csv(path)

# Inspect head
loans.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,spread,issue_yr_mnth,avg_spread,min_spread,return,beta,disc_rate,loan_duration,avg_monthly_cash_flow,rar
0,49763530,NaN,7000.0,7000.0,7000.0,36 months,0.0789,219.00,A,A5,...,0.077472,2015-05,0.124520,0.051806,0.124084,0.404783,0.006530,32.0,245.893361,7126.312861
1,1614412,NaN,5000.0,5000.0,5000.0,36 months,0.0603,152.18,A,A1,...,0.058818,2012-10,0.137318,0.058518,0.091950,0.404783,0.007239,29.0,188.267241,4944.316301
2,19435976,NaN,10000.0,10000.0,10000.0,36 months,0.0712,309.32,A,A3,...,0.070246,2014-06,0.138971,0.059056,0.113552,0.404783,0.007315,36.0,309.319915,9830.124029
3,44756872,NaN,12000.0,12000.0,12000.0,36 months,0.0593,364.69,A,A1,...,0.057898,2015-04,0.124802,0.057898,0.093717,0.404783,0.006820,36.0,364.572381,11681.851552
4,108855699,NaN,6700.0,6700.0,6700.0,36 months,0.0532,201.77,A,A1,...,0.042755,2017-05,0.126481,0.042755,0.029875,0.404783,0.006173,8.0,862.520378,6753.795702


In [ ]:
# Drop intermediate features used to derive RAR
cols = [
    "issue_yr",
    "avg_cdi_rate",
    "spread",
    "issue_yr_mnth",
    "avg_spread",
    "min_spread",
    "return",
    "beta",
    "disc_rate",
    "loan_duration",
    "avg_monthly_cash_flow"
]

loans.drop(cols, axis=1, inplace=True)

In [ ]:
loans.info(verbose=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 268615 entries, 0 to 268614
Data columns (total 152 columns):
 #    Column                                      Dtype  
---   ------                                      -----  
 0    id                                          int64  
 1    member_id                                   float64
 2    loan_amnt                                   float64
 3    funded_amnt                                 float64
 4    funded_amnt_inv                             float64
 5    term                                        object 
 6    int_rate                                    float64
 7    installment                                 float64
 8    grade                                       object 
 9    sub_grade                                   object 
 10   emp_title                                   object 
 11   emp_length                                  object 
 12   home_ownership                              object 
 13   annual_inc  

## Data Cleaning & Feature Engineering

In [ ]:
# 1. Screen out uninformative features

# ID zero-variance features
zero_var_cols = [
    col for col in loans.columns
    if loans[col].nunique(dropna=True) <= 1
]

print(f"Zero-variance features ({len(zero_var_cols)}): {zero_var_cols}")

# ID high-cardinality textual, categorical, and datetime features
high_card_cols = [
    col for col in loans.columns
    if loans[col].dtype == "object"
    and loans[col].nunique(dropna=True) > 50  # Adjusted threshold from 100 to 50 to align with M&K's post-DPT feature count
]

print(f"\nHigh-cardinality features ({len(high_card_cols)}): {high_card_cols}")

# Specify relevant datetime features to preserve for decomposition
date_cols = ["issue_d", "earliest_cr_line"]

# Specify remaining logically irrelevant features
irr_cols = ["id"]

# Combine identified uninformative features
uninf_cols = list(set(
    zero_var_cols
    + [col for col in high_card_cols if col not in date_cols]
    + irr_cols
))

print(f"\nTotal uninformative features: {len(uninf_cols)}")

# Drop uninformative features
loans.drop(columns=uninf_cols, inplace=True)
print(f"\nShape after dropping uninformative features: {loans.shape[0]:,} rows, {loans.shape[1]} columns")

Zero-variance features (10): ['member_id', 'pymnt_plan', 'out_prncp', 'out_prncp_inv', 'next_pymnt_d', 'policy_code', 'hardship_flag', 'hardship_type', 'deferral_term', 'hardship_length']

High-cardinality features (13): ['emp_title', 'issue_d', 'url', 'desc', 'title', 'zip_code', 'addr_state', 'earliest_cr_line', 'last_pymnt_d', 'last_credit_pull_d', 'sec_app_earliest_cr_line', 'debt_settlement_flag_date', 'settlement_date']

Total uninformative features: 22

Shape after dropping uninformative features: 268,615 rows, 130 columns


In [ ]:
# 2. Check for duplicate observations
loans.duplicated().sum()

np.int64(0)

In [ ]:
# 3. Handle datetime typing and feature engineering

# Convert relevant date string features to datetime type
loans["issue_d"] = pd.to_datetime(loans["issue_d"])
loans["earliest_cr_line"] = pd.to_datetime(loans["earliest_cr_line"], format="%b-%Y")

# Extract issue year
loans["issue_yr"] = loans["issue_d"].dt.year

# Compute length of credit history in months
loans["cr_hist_len"] = (
    (loans["issue_yr"] - loans["earliest_cr_line"].dt.year) * 12
    + (loans["issue_d"].dt.month - loans["earliest_cr_line"].dt.month) + 1
)

# Drop original date columns
loans.drop(columns=date_cols, inplace=True)

print(f"Shape after datetime handling: {loans.shape[0]:,} rows, {loans.shape[1]} columns")
print(f"\nissue_yr summary stats:")
print(loans["issue_yr"].describe())
print(f"\ncr_hist_len summary stats:")
print(loans["cr_hist_len"].describe())

Shape after datetime handling: 268,615 rows, 130 columns

issue_yr summary stats:
count    268615.000000
mean       2014.972314
std           1.641605
min        2007.000000
25%        2014.000000
50%        2015.000000
75%        2016.000000
max        2018.000000
Name: issue_yr, dtype: float64

cr_hist_len summary stats:
count    268615.00000
mean        196.00695
std          90.08429
min          37.00000
25%         136.00000
50%         178.00000
75%         241.00000
max         823.00000
Name: cr_hist_len, dtype: float64


In [ ]:
# 4. Discretize mths_since_last_delinq (required for clustering)
def get_delinq_recency_bin(row):
  mths_since_last_delinq = row["mths_since_last_delinq"]

  if pd.isna(mths_since_last_delinq):
    return "Never Delinquent"  # Assuming null values indicate absence of historical delinquency
  elif mths_since_last_delinq < 12.0:
    return "< 12 Months"
  elif mths_since_last_delinq < 36.0:
    return "12-36 Months"
  else:
    return "> 36 Months"

loans["delinq_recency_bin"] = loans.apply(get_delinq_recency_bin, axis=1)

In [ ]:
# 5. Handle mistyped numeric features

# Convert term string to integer
term_map = {" 36 months": 36, " 60 months": 60}
loans["term"] = loans["term"].map(term_map)

# Convert emp_length string to integer
emp_length_map = {
    "< 1 year": 0,
    "1 year": 1,
    "2 years": 2,
    "3 years": 3,
    "4 years": 4,
    "5 years": 5,
    "6 years": 6,
    "7 years": 7,
    "8 years": 8,
    "9 years": 9,
    "10+ years": 10
}

loans["emp_length"] = loans["emp_length"].map(emp_length_map)

In [ ]:
# 6. Implement preliminary missing value treatment strategy
# Note: Saving imputation for preprocessing pipeline to prevent cross-validation data leakage

# Define function to identify columns with missingness above a specified threshold
def check_missingness(df, threshold):
  """
  Identifies and returns sparse columns in a provided DataFrame with missingness above a specified threshold.

  Parameters:
  -----------
      - df (pandas.core.frame.DataFrame): DataFrame to analyze.
      - threshold (float): Proportion of missing values above which a column is considered sparse.

  Returns:
  --------
      - dict: Dictionary containing column names as keys and corresponding missingness percentages as values.
  """
  # Get total observation count
  n_obs = len(df)

  # Initialize empty dict to store sparse column names and respective missingness %s
  sparse_cols = {}

  # Iterate through columns
  for col in df.columns:
    # Compute missingness %
    pct_missing = df[col].isna().sum() / n_obs

    # Check if missingness % is above threshold
    if pct_missing > threshold:
      sparse_cols[col] = pct_missing

  # Sort dict desc
  sparse_cols = dict(sorted(sparse_cols.items(), key=lambda x: x[1], reverse=True))

  # Return dict
  return sparse_cols

# ID and drop features with extensive (> 50%) missingness
sparse_cols = check_missingness(loans, 0.5)

print("=" * 40, "\nFeature Removal", "\n" + "=" * 40)
print("\nSparsely-populated features:")
for i, (col, pct) in enumerate(sparse_cols.items(), start=1):
  print(f"{i}. {col}: {pct:.2%}")

loans.drop(columns=list(sparse_cols.keys()), inplace=True)
print(f"\nShape after dropping sparsely-populated features: {loans.shape[0]:,} rows, {loans.shape[1]} columns")

# ID and drop observations with > 5% of features missing
n_features = len(loans.columns)
missing_thresh = np.floor(0.05 * n_features)

mask = loans.isna().sum(axis=1) > missing_thresh
n_obs_dropped = mask.sum()
loans = loans[~mask]

print("\n" + "=" * 40, "\nObservation Omission", "\n" + "=" * 40)
print(f"\nShape after dropping rows with > 5% of features missing: {loans.shape[0]:,} rows, {loans.shape[1]} columns")
print(f"\nRows dropped: {n_obs_dropped:,}")

# ID features with lingering missingness
# These will be handled via imputation within the preprocessing pipeline
missing_s = loans.isna().sum()
missing_s = missing_s[missing_s > 0].sort_values(ascending=False)
print("\n" + "=" * 40, "\nRemaining Missingness", "\n" + "=" * 40)
print(f"\nFeatures with remaining missingness:")
print(missing_s)

Feature Removal 

Sparsely-populated features:
1. orig_projected_additional_accrued_interest: 99.73%
2. hardship_reason: 99.58%
3. hardship_status: 99.58%
4. hardship_amount: 99.58%
5. hardship_start_date: 99.58%
6. hardship_end_date: 99.58%
7. payment_plan_start_date: 99.58%
8. hardship_dpd: 99.58%
9. hardship_loan_status: 99.58%
10. hardship_payoff_balance_amount: 99.58%
11. hardship_last_payment_amount: 99.58%
12. sec_app_mths_since_last_major_derog: 99.52%
13. sec_app_revol_util: 98.65%
14. revol_bal_joint: 98.63%
15. sec_app_fico_range_low: 98.63%
16. sec_app_fico_range_high: 98.63%
17. sec_app_inq_last_6mths: 98.63%
18. sec_app_mort_acc: 98.63%
19. sec_app_open_acc: 98.63%
20. sec_app_open_act_il: 98.63%
21. sec_app_num_rev_accts: 98.63%
22. sec_app_chargeoff_within_12_mths: 98.63%
23. sec_app_collections_12_mths_ex_med: 98.63%
24. verification_status_joint: 98.10%
25. dti_joint: 98.09%
26. annual_inc_joint: 98.09%
27. settlement_status: 97.52%
28. settlement_amount: 97.52%
29. s

In [ ]:
# 7. Handle multicollinearity via correlation-based testing and redundant feature removal

def get_high_corr_pairs(df, target_col, corr_thresh=0.9):
    """
    Identifies and returns a DataFrame containing numeric feature pairs with absolute correlation above a specified threshold.

    Parameters:
    -----------
        - df (pandas.core.frame.DataFrame): DataFrame to analyze.
        - target_col (str): Name of the target column to compute correlation with.
        - corr_thresh (float): Absolute correlation threshold above which multicollinearity is considered an issue. Default is 0.9.

    Returns:
    --------
        - pandas.core.frame.DataFrame: DataFrame containing feature pairs with absolute correlation above the specified threshold.
    """
    # Get list of numeric columns
    num_cols = df.select_dtypes(include="number").columns.tolist()

    # Get correlation matrix
    corr_matrix = df[num_cols].corr().abs()

    # Isolate target correlations
    target_corrs = corr_matrix[target_col]

    # Isolate pairwise feature correlations
    feat_corrs = corr_matrix.drop(index=target_col, columns=target_col)

    # Extract upper triangle of feature correlation matrix to prevent redundancy
    upper_tri = feat_corrs.where(np.triu(np.ones(feat_corrs.shape), k=1).astype(bool))

    # ID feature pairs where correlation exceeds threshold
    high_corr_indices = np.where(upper_tri > corr_thresh)

    high_corr_pairs = []
    for i, j in zip(*high_corr_indices):
      feat1 = feat_corrs.index[i]
      feat2 = feat_corrs.columns[j]

      high_corr_pairs.append({
          "Feature 1 Name": feat1,
          "Feature 1 Target Correlation": target_corrs.loc[feat1],
          "Feature 2 Name": feat2,
          "Feature 2 Target Correlation": target_corrs.loc[feat2],
          "Pairwise Correlation": upper_tri.iloc[i, j].round(4)
      })

    # Handle edge case with no high-correlation pairs
    if not high_corr_pairs:
      return pd.DataFrame(columns=[
          "Feature 1 Name",
          "Feature 1 Target Correlation",
          "Feature 2 Name",
          "Feature 2 Target Correlation",
          "Pairwise Correlation"
      ])

    return pd.DataFrame(high_corr_pairs).sort_values("Pairwise Correlation", ascending=False)

# ID and print numeric feature pairs with correlation > 0.9
corr_thresh = 0.9
high_corr_df = get_high_corr_pairs(loans, "rar", corr_thresh)

print(f"Number of highly correlated feature pairs (|r| > {corr_thresh}): {len(high_corr_df)}")
print("\nHighly correlated feature pairs:")
print(high_corr_df.to_string(index=False))

# Specify redundant features to drop based on domain knowledge and/or correlation with target
redundant_cols = [
    "loan_amnt",
    "funded_amnt_inv",
    "fico_range_high",
    "total_pymnt_inv",
    "open_acc",
    "num_actv_rev_tl",
    "collection_recovery_fee",
    "tot_cur_bal",
    "total_rec_prncp",
    "installment",
    "cr_hist_len"
]

# Drop redundant features
loans.drop(columns=redundant_cols, inplace=True)
print(f"\nShape after redundant feature removal: {loans.shape[0]:,} rows, {loans.shape[1]} columns")

# Confirm multicollinearity resolved
high_corr_df_post = get_high_corr_pairs(loans, "rar", corr_thresh)
print(f"\nNumber of highly correlated feature pairs remaining (|r| > {corr_thresh}): {len(high_corr_df_post)}")
if len(high_corr_df_post) == 0:
    print("Success: All specified multicollinearity has been resolved.")
else:
    print("\nRemaining highly correlated feature pairs:")
    print(high_corr_df_post.to_string(index=False))

Number of highly correlated feature pairs (|r| > 0.9): 15

Highly correlated feature pairs:
      Feature 1 Name  Feature 1 Target Correlation          Feature 2 Name  Feature 2 Target Correlation  Pairwise Correlation
           loan_amnt                      0.868009             funded_amnt                      0.868016                1.0000
           loan_amnt                      0.868009         funded_amnt_inv                      0.868019                1.0000
         funded_amnt                      0.868016         funded_amnt_inv                      0.868019                1.0000
      fico_range_low                      0.128390         fico_range_high                      0.128389                1.0000
         total_pymnt                      0.990244         total_pymnt_inv                      0.990247                1.0000
            open_acc                      0.155630                num_sats                      0.156394                0.9987
     num_actv_rev_t

> **Feature Removal Logic:**
> * `loan_amnt` was dropped because it represents the amount *applied for* by the borrower, whereas `funded_amnt` represents the actual amount *granted to* the borrower and has a slightly higher correlation with the target
> * `funded_amnt_inv` was dropped because it is a component/constituent of the aggregate `funded_amnt`
> * `fico_range_high` was dropped because `fico_range_low` has a slightly higher correlation with the target
> * `total_pymnt_inv` was dropped because it is a component/constituent of the aggregate `total_pymnt`
> * `open_acc` was dropped because `num_sats` has a slightly higher correlation with the target
> * `num_actv_rev_tl` was dropped because `num_rev_tl_bal_gt_0` has a slightly higher correlation with the target
> * `collection_recovery_fee` was dropped because `recoveries` has a slightly higher correlation with the target
> * `tot_cur_bal` was dropped because `tot_hi_cred_lim` has a slightly higher correlation with the target
> * `total_rec_prncp` was dropped because it is a component/constituent of the aggregate `total_pymnt`, which also has a slightly higher correlation with the target
> * `installment` was dropped because `funded_amnt` has a slightly higher correlation with the target
> * `cr_hist_len` was dropped because it is analogous to `mo_sin_old_rev_tl_op`, which also has a slightly higher correlation with the target

In [ ]:
# Get cleaned dataset summary
loans.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 254328 entries, 0 to 268614
Data columns (total 71 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   funded_amnt                 254328 non-null  float64
 1   term                        254328 non-null  int64  
 2   int_rate                    254328 non-null  float64
 3   grade                       254328 non-null  object 
 4   sub_grade                   254328 non-null  object 
 5   emp_length                  239535 non-null  float64
 6   home_ownership              254328 non-null  object 
 7   annual_inc                  254328 non-null  float64
 8   verification_status         254328 non-null  object 
 9   loan_status                 254328 non-null  object 
 10  purpose                     254328 non-null  object 
 11  dti                         254265 non-null  float64
 12  delinq_2yrs                 254328 non-null  float64
 13  fico_range_low     

## Data Export

In [ ]:
# Define path
path = BASE_PATH + "p2p-customer-value-prediction/data/processed/cleaned_loans.csv"

# Write cleaned dataset to CSV
loans.to_csv(path, index=False)